In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
import numpy as np
import pandas as pd
import xgboost as xgb


C:\Users\Owner\Downloads\Datamining_final\.venv\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
df = pd.read_pickle('df_with_embeddings.pkl')
df['simple_bias'] = 'center'
df.loc[(df['bias'] == 'leaning-left') | (df['bias'] == 'left'), 'simple_bias'] = 'left'
df.loc[(df['bias'] == 'leaning-right') | (df['bias'] == 'right'), 'simple_bias'] = 'right'
embeddings = np.stack(df['embeddings'].values)
le = LabelEncoder()
y = le.fit_transform(df['simple_bias'])  # left=0, center=1, right=2

X_train, X_test, y_train, y_test = train_test_split(
    embeddings, y, test_size=0.2, random_state=42, stratify=y
)

In [3]:
baseline = LogisticRegression(max_iter=1000, random_state=42)
baseline.fit(X_train, y_train)
print("Baseline Logistic Regression:")
print(classification_report(y_test, baseline.predict(X_test), target_names=le.classes_))

Baseline Logistic Regression:
              precision    recall  f1-score   support

      center       0.73      0.09      0.16       296
        left       0.63      0.84      0.72       976
       right       0.73      0.67      0.69       895

    accuracy                           0.67      2167
   macro avg       0.69      0.53      0.52      2167
weighted avg       0.68      0.67      0.63      2167



In [4]:
def xgb_objective(params):
    model = xgb.XGBClassifier(
        n_estimators=int(params['n_estimators']),
        max_depth=int(params['max_depth']),
        learning_rate=params['learning_rate'],
        subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        eval_metric='mlogloss',
        random_state=42,
        device='cuda'  # remove if no GPU
    )
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='f1_macro').mean()
    return {'loss': -score, 'status': STATUS_OK}

xgb_space = {
    'n_estimators': hp.quniform('n_estimators', 100, 500, 50),
    'max_depth': hp.quniform('max_depth', 3, 10, 1),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),
    'subsample': hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
}

xgb_trials = Trials()
xgb_best = fmin(fn=xgb_objective, space=xgb_space, algo=tpe.suggest,
                max_evals=30, trials=xgb_trials)

print("Best XGBoost params:", xgb_best)

  0%|          | 0/30 [00:00<?, ?trial/s, best loss=?]

C:\Users\Owner\Downloads\Datamining_final\.venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [02:47:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



100%|██████████| 30/30 [11:27<00:00, 22.92s/trial, best loss: -0.7117137906503936]
Best XGBoost params: {'colsample_bytree': np.float64(0.884255481785085), 'learning_rate': np.float64(0.2845545180160818), 'max_depth': np.float64(4.0), 'n_estimators': np.float64(400.0), 'subsample': np.float64(0.6465384145160572)}


In [5]:
def lr_objective(params):
    model = LogisticRegression(
        C=params['C'],
        solver=params['solver'],
        max_iter=1000,
        random_state=42
    )
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='f1_macro').mean()
    return {'loss': -score, 'status': STATUS_OK}

lr_space = {
    'C': hp.loguniform('C', np.log(0.001), np.log(100)),
    'solver': hp.choice('solver', ['lbfgs', 'saga'])
}

lr_trials = Trials()
lr_best = fmin(fn=lr_objective, space=lr_space, algo=tpe.suggest,
               max_evals=20, trials=lr_trials)

print("Best LR params:", lr_best)

100%|██████████| 20/20 [02:26<00:00,  7.30s/trial, best loss: -0.7074799101228958]
Best LR params: {'C': np.float64(75.49950410337986), 'solver': np.int64(1)}


In [6]:
best_xgb = xgb.XGBClassifier(
    n_estimators=int(xgb_best['n_estimators']),
    max_depth=int(xgb_best['max_depth']),
    learning_rate=xgb_best['learning_rate'],
    subsample=xgb_best['subsample'],
    colsample_bytree=xgb_best['colsample_bytree'],
    eval_metric='mlogloss',
    random_state=42
)
best_xgb.fit(X_train, y_train)
print("\nTuned XGBoost:")
print(classification_report(y_test, best_xgb.predict(X_test), target_names=le.classes_))

best_lr = LogisticRegression(C=lr_best['C'], max_iter=1000, random_state=42)
best_lr.fit(X_train, y_train)
print("\nTuned Logistic Regression:")
print(classification_report(y_test, best_lr.predict(X_test), target_names=le.classes_))


Tuned XGBoost:
              precision    recall  f1-score   support

      center       0.79      0.48      0.60       296
        left       0.74      0.86      0.79       976
       right       0.81      0.77      0.79       895

    accuracy                           0.77      2167
   macro avg       0.78      0.70      0.73      2167
weighted avg       0.77      0.77      0.76      2167


Tuned Logistic Regression:
              precision    recall  f1-score   support

      center       0.76      0.50      0.61       296
        left       0.73      0.83      0.78       976
       right       0.80      0.77      0.79       895

    accuracy                           0.76      2167
   macro avg       0.77      0.70      0.72      2167
weighted avg       0.77      0.76      0.76      2167

